# Run Full TurboQuant Benchmark Suite

This notebook runs the same core benchmark suite as `run_all_benchmarks.sh`:

1. `benchmarks/bench_kernels.py`
2. `benchmarks/bench_tq_attention.py`
3. `baselines/fp16_baseline.py`
4. `baselines/fp8_baseline.py`
5. `baselines/int4_baseline.py`
6. `benchmarks/bench_quality.py`
7. `analysis/plot_results.py`

It is designed for MI300X and writes logs to `results/logs/` and JSON outputs to `results/`.

**Reporting:** regenerate all v2 figures (including kv-heavy rocprof **Fig 30** and engineering-closure **Fig 31**) via `python3 report/generate_figures_v2.py`. For the written account of *everything we implemented vs what must move on the ROCm+vLLM image*, see `docs/repo_decode_bottleneck_closure.md` and report **§14** / **§14.5** / paper **§5.13**.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys
from datetime import datetime

root = Path.cwd().resolve()
if not (root / "benchmarks").exists() and (root.parent / "benchmarks").exists():
    root = root.parent

assert (root / "benchmarks").exists(), f"Could not find project root from {Path.cwd()}"
log_dir = root / "results" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

MODEL = "mistralai/Mistral-7B-v0.1"
SEQ_LENS = [512, 2048, 8192, 32768, 65536, 131072]
N_DECODE = 30
N_RUNS = 3

print("root:", root)
print("model:", MODEL)
print("seq_lens:", SEQ_LENS)
print("n_decode:", N_DECODE, "n_runs:", N_RUNS)
print("log_dir:", log_dir)

In [ ]:
def run_cmd(args, log_name):
    log_path = log_dir / log_name
    printable = " ".join(shlex.quote(str(a)) for a in args)
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {printable}")
    with log_path.open("w") as logf:
        proc = subprocess.Popen(
            args,
            cwd=str(root),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            logf.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {printable}")
    print(f"  -> log saved: {log_path}\n")

In [ ]:
cmds = [
    ([sys.executable, "benchmarks/bench_kernels.py", "--n-vectors", "65536", "--n-iters", "100"], "bench_kernels.log"),
    ([sys.executable, "benchmarks/bench_tq_attention.py", "--seq-lens", *[str(x) for x in SEQ_LENS], "--n-iters", "20"], "bench_tq_attention.log"),
    ([sys.executable, "baselines/fp16_baseline.py", "--model", MODEL, "--seq-lens", *[str(x) for x in SEQ_LENS], "--n-decode", str(N_DECODE), "--n-runs", str(N_RUNS)], "fp16_baseline.log"),
    ([sys.executable, "baselines/fp8_baseline.py", "--model", MODEL, "--seq-lens", *[str(x) for x in SEQ_LENS], "--n-decode", str(N_DECODE), "--n-runs", str(N_RUNS)], "fp8_baseline.log"),
    ([sys.executable, "baselines/int4_baseline.py", "--model", MODEL, "--seq-lens", *[str(x) for x in SEQ_LENS], "--n-decode", str(N_DECODE), "--n-runs", str(N_RUNS)], "int4_baseline.log"),
    ([sys.executable, "benchmarks/bench_quality.py", "--model", MODEL, "--n-tokens", "4096", "--context-len", "512"], "bench_quality.log"),
    ([sys.executable, "analysis/plot_results.py"], "plot_results.log"),
]

for args, log_name in cmds:
    run_cmd(args, log_name)

print("All benchmark steps completed.")

In [ ]:
expected = [
    root / "results" / "bench_kernels.json",
    root / "results" / f"fp16_baseline_{MODEL.replace('/', '_')}.json",
    root / "results" / f"fp8_baseline_{MODEL.replace('/', '_')}.json",
    root / "results" / f"int4_baseline_{MODEL.replace('/', '_')}.json",
    root / "results" / f"bench_quality_{MODEL.replace('/', '_')}.json",
    root / "analysis" / "figures" / "throughput_vs_context.png",
]

for p in expected:
    print("OK" if p.exists() else "MISSING", p)